# 03 — Tier 1: Baseline (TF-IDF + Classical ML)

Strategy:
- **Aspect Detection**: Multi-label TF-IDF + Binary Relevance (one classifier per aspect)
- **Sentiment Classification**: Per-aspect TF-IDF + Logistic Regression
- **Heuristics**: Use star_rating + platform as auxiliary features
- Fast to train, interpretable, good fallback for low-resource aspects

Inputs: `data/processed/train_clean.csv`, `data/processed/val_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
import json, ast
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import joblib

VALID_ASPECTS    = ['food','service','price','cleanliness','delivery',
                    'ambiance','app_experience','general','none']
VALID_SENTIMENTS = ['positive','negative','neutral']
PROC_DIR = Path('data/processed')
MODEL_DIR = Path('models/tier1')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Setup ✓')

## 1. Load Processed Data

In [ ]:
def safe_parse(x):
    if isinstance(x, (list, dict)): return x
    try: return ast.literal_eval(str(x))
    except: return None

train = pd.read_csv(PROC_DIR / 'train_clean.csv')
val   = pd.read_csv(PROC_DIR / 'val_clean.csv')
hidden = pd.read_csv(PROC_DIR / 'hidden_test_clean.csv')

for df in [train, val]:
    df['aspects']           = df['aspects'].apply(safe_parse)
    df['aspect_sentiments'] = df['aspect_sentiments'].apply(safe_parse)

print('Train:', len(train), '| Val:', len(val), '| Hidden:', len(hidden))

## 2. Aspect Detection — Multi-label Classification

In [ ]:
# --- Prepare multi-label targets ---
mlb = MultiLabelBinarizer(classes=VALID_ASPECTS)

y_train = mlb.fit_transform(train['aspects'].apply(
    lambda x: [a for a in (x or []) if a in VALID_ASPECTS]))
y_val   = mlb.transform(val['aspects'].apply(
    lambda x: [a for a in (x or []) if a in VALID_ASPECTS]))

print('y_train shape:', y_train.shape)
print('Aspect label columns:', mlb.classes_.tolist())

In [ ]:
# --- TF-IDF on cleaned Arabic text ---
tfidf_aspect = TfidfVectorizer(
    analyzer='char_wb',    # char n-grams handle Arabic morphology well
    ngram_range=(2, 5),
    max_features=60_000,
    sublinear_tf=True,
    min_df=2
)

X_train_tfidf = tfidf_aspect.fit_transform(train['text_clean'].fillna(''))
X_val_tfidf   = tfidf_aspect.transform(val['text_clean'].fillna(''))
X_hidden_tfidf = tfidf_aspect.transform(hidden['text_clean'].fillna(''))

# Add auxiliary features: star_rating_norm, is_app_platform
def aux_features(df):
    return csr_matrix(np.column_stack([
        df['star_rating_norm'].fillna(0.6).values,
        df['is_app_platform'].fillna(0).values,
        df['arabic_ratio'].fillna(0).values if 'arabic_ratio' in df.columns else np.zeros(len(df))
    ]))

X_train = hstack([X_train_tfidf, aux_features(train)])
X_val   = hstack([X_val_tfidf,   aux_features(val)])
X_hidden = hstack([X_hidden_tfidf, aux_features(hidden)])

print('Feature matrix shape (train):', X_train.shape)

In [ ]:
# --- Train One-vs-Rest classifier ---
aspect_clf = OneVsRestClassifier(
    LogisticRegression(C=1.0, max_iter=500, solver='lbfgs', class_weight='balanced'),
    n_jobs=-1
)
aspect_clf.fit(X_train, y_train)

# Evaluate
y_pred_val = aspect_clf.predict(X_val)
micro_f1 = f1_score(y_val, y_pred_val, average='micro', zero_division=0)
print(f'Aspect Detection — Micro F1 on Val: {micro_f1:.4f}')

# Per-class F1
per_class = f1_score(y_val, y_pred_val, average=None, zero_division=0)
for asp, f in zip(mlb.classes_, per_class):
    print(f'  {asp:<18} F1={f:.3f}')

## 3. Sentiment Classification — Per-Aspect

In [ ]:
# --- Load flat data ---
train_flat = pd.read_csv(PROC_DIR / 'train_flat.csv')
val_flat   = pd.read_csv(PROC_DIR / 'val_flat.csv')

# Build one sentiment classifier per aspect (handles aspect-specific vocab)
sentiment_clfs = {}
sentiment_tfidf = {}

for asp in VALID_ASPECTS:
    tr = train_flat[train_flat['aspect'] == asp]
    vl = val_flat[val_flat['aspect'] == asp]
    
    if len(tr) < 5:
        print(f'  Skipping {asp} (too few samples: {len(tr)})')
        continue
    
    tfidf = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(2, 5),
        max_features=30_000, sublinear_tf=True, min_df=1
    )
    X_tr = tfidf.fit_transform(tr['text_clean'].fillna(''))
    y_tr = tr['sentiment'].values
    
    clf = LogisticRegression(C=1.0, max_iter=500, class_weight='balanced', solver='lbfgs')
    clf.fit(X_tr, y_tr)
    
    if len(vl) > 0:
        X_vl = tfidf.transform(vl['text_clean'].fillna(''))
        y_vl = vl['sentiment'].values
        f1 = f1_score(y_vl, clf.predict(X_vl), average='micro', zero_division=0)
        print(f'  {asp:<18} val_micro_f1={f1:.3f}  (n_train={len(tr)})')
    
    sentiment_clfs[asp]  = clf
    sentiment_tfidf[asp] = tfidf

print('\nSentiment classifiers trained ✓')

## 4. Star Rating Heuristic Fallback

In [ ]:
def rating_to_sentiment(star_rating):
    """Heuristic fallback when no model available for an aspect."""
    try:
        r = float(star_rating)
    except:
        return 'neutral'
    if r >= 4:
        return 'positive'
    elif r <= 2:
        return 'negative'
    return 'neutral'

print('Heuristic fallback ready ✓')

## 5. Full Inference Pipeline

In [ ]:
def predict_review(text_clean: str, star_rating=None, is_app_platform: int = 0,
                   arabic_ratio: float = 0.8, threshold: float = 0.3) -> dict:
    """
    Returns {'aspects': [...], 'aspect_sentiments': {...}}
    """
    # Step 1: Detect aspects
    tfidf_feat = tfidf_aspect.transform([text_clean])
    aux        = csr_matrix([[star_rating / 5.0 if star_rating else 0.6,
                               is_app_platform, arabic_ratio]])
    feat = hstack([tfidf_feat, aux])
    
    proba = aspect_clf.predict_proba(feat)[0]  # shape (n_aspects,)
    detected = [asp for asp, p in zip(mlb.classes_, proba) if p >= threshold]
    
    # If nothing detected, use top-1
    if not detected:
        detected = [mlb.classes_[np.argmax(proba)]]
    
    # Step 2: Sentiment per detected aspect
    sentiments = {}
    for asp in detected:
        if asp in sentiment_clfs:
            x = sentiment_tfidf[asp].transform([text_clean])
            sentiments[asp] = sentiment_clfs[asp].predict(x)[0]
        else:
            sentiments[asp] = rating_to_sentiment(star_rating)
    
    return {'aspects': detected, 'aspect_sentiments': sentiments}

# Quick sanity check
test_text = 'الاكل كان ممتاز بس الخدمه بطيئه والسعر غالي'
result = predict_review(test_text, star_rating=3, arabic_ratio=0.9)
print('Test prediction:', result)

## 6. Generate Hidden Test Predictions

In [ ]:
predictions = []
for _, row in hidden.iterrows():
    pred = predict_review(
        text_clean    = str(row.get('text_clean', '')),
        star_rating   = row.get('star_rating'),
        is_app_platform = int(row.get('is_app_platform', 0)),
        arabic_ratio  = float(row.get('arabic_ratio', 0.8))
    )
    predictions.append({
        'review_id'         : int(row['review_id']),
        'aspects'           : pred['aspects'],
        'aspect_sentiments' : pred['aspect_sentiments']
    })

out_path = Path('predictions/tier1_predictions.json')
out_path.parent.mkdir(exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f'Saved {len(predictions)} predictions → {out_path}')
print('Sample prediction:', predictions[0])

## 7. Save Models

In [ ]:
joblib.dump(tfidf_aspect,   MODEL_DIR / 'tfidf_aspect.pkl')
joblib.dump(aspect_clf,     MODEL_DIR / 'aspect_clf.pkl')
joblib.dump(mlb,            MODEL_DIR / 'mlb.pkl')
joblib.dump(sentiment_clfs, MODEL_DIR / 'sentiment_clfs.pkl')
joblib.dump(sentiment_tfidf,MODEL_DIR / 'sentiment_tfidf.pkl')
print('All models saved to', MODEL_DIR)

## 8. Validation Evaluation (End-to-End)

In [ ]:
# Compute end-to-end ABSA micro F1 on validation set
def compute_absa_f1(df):
    """
    Micro F1 over all (review_id, aspect, sentiment) tuples.
    This mirrors the competition scoring.
    """
    true_tuples, pred_tuples = set(), set()
    
    for _, row in df.iterrows():
        rid = row['review_id']
        gt_sents = row['aspect_sentiments'] if isinstance(row['aspect_sentiments'], dict) else {}
        for asp, sent in gt_sents.items():
            true_tuples.add((rid, asp, sent))
        
        pred = predict_review(
            str(row.get('text_clean', '')),
            star_rating=row.get('star_rating'),
            is_app_platform=int(row.get('is_app_platform', 0)),
            arabic_ratio=float(row.get('arabic_ratio', 0.8))
        )
        for asp, sent in pred['aspect_sentiments'].items():
            pred_tuples.add((rid, asp, sent))
    
    tp = len(true_tuples & pred_tuples)
    precision = tp / max(len(pred_tuples), 1)
    recall    = tp / max(len(true_tuples), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'n_true': len(true_tuples), 'n_pred': len(pred_tuples), 'tp': tp}

metrics = compute_absa_f1(val)
print('=== End-to-End ABSA Evaluation (Val) ===')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')